In [ ]:
import pickle
import time
import warnings
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse
import shapely
import tqdm
import quadpy
from pathlib import Path
import warnings
from random_matrix.amplitude_matrix import (
    isotropic_sphere,
    scattering_geometry,
)
from random_matrix.input_statistics import density_function, density_integrals
from random_matrix.input_statistics.density_function import (
    DeltaDensityFactor,
    DensityFunction,
    DensityFunctionTerm,
    RegularDensityFactor,
)
from random_matrix.input_statistics.index_finder import IndexFinder
from random_matrix.input_statistics.input_statistics_manager import (
    InputStatisticsManager,
)
from random_matrix.input_statistics.integration_task import (
    IntegrationTaskPreparer,
    IntegrationTaskConfig,
)
from random_matrix.input_statistics.medium_parameters import MediumParameters
from random_matrix.input_statistics.medium_statistics import (
    MediumStatistics,
    ParticleStatistics,
)
from random_matrix.modes import mode_grid, mode_grid_factory
from random_matrix.utils import (
    array_utils,
    function_utils,
    geometry_utils,
    integration_utils,
    matrix_utils,
    special_functions,
)
from random_matrix.scattering_matrix import sampler

In [ ]:
MODE_GRIDS_FILENAME = Path(
    "/home/nbyrnes/code/random-matrix/paper_data/mode_grids.pkl"
)
with MODE_GRIDS_FILENAME.open("rb") as f:
    mode_grids = pickle.load(f)

In [ ]:
first = [
    "pp,pp",
    "pp,pe",
    "pp,ep",
    "pp,ee",
    "pe,pe",
    "pe,pe",
    "pe,ep",
    "pe,ee",
    "ep,ep",
    "ep,ee",
    "ee,ee",
]
second = [
    "t,t",
    "t,r",
    "t,t2",
    "t,r2",
    "r,r",
    "r,t2",
    "r,r2",
    "t2,t2",
    "t2,r2",
    "r2,r2",
]

In [ ]:
my_grid = mode_grids[0]

wavelength = 550e-9
slab_thickness = 1.8992695221776513e-06
number_density = 5.921762640653617e17
medium_parameters = MediumParameters(
    wavelength=wavelength,
    number_density=number_density,
    slab_thickness=slab_thickness,
)
term = DensityFunctionTerm.from_delta({"x": 2.0, "m": 1.2})

# 2D version
particle_statistics = ParticleStatistics(
    term,
    isotropic_sphere.get_A,
    isotropic_sphere.get_A_product,
    isotropic_sphere.get_A_product_conj,
)
medium_statistics = MediumStatistics([particle_statistics])


# Set up the relevant indices
propagating_indices = my_grid.propagating_indices
quads = [(0, i, 0, i) for i in propagating_indices]
quads = [(0, 0, 0, 0), (0, 5, 0, 5), (1, 2, 3, 4), (3, 4, 1, 2)]
supplied_indices = {
    "covariance": {key1: {key2: [] for key2 in second} for key1 in first}
}
for key in ["t,t"]:
    supplied_indices["covariance"]["pp,pp"][key] = quads

In [ ]:
supplied_indices["covariance"]["pp,pp"]

{'t,t': [(0, 0, 0, 0), (0, 5, 0, 5), (1, 2, 3, 4), (3, 4, 1, 2)],
 't,r': [],
 't,t2': [],
 't,r2': [],
 'r,r': [],
 'r,t2': [],
 'r,r2': [],
 't2,t2': [],
 't2,r2': [],
 'r2,r2': []}

In [ ]:
# 2D NUMPY
use_np_config = IntegrationTaskConfig(use_gpu=False)

simulation_name = f"test_one"
input_statistics_manager = InputStatisticsManager(
    simulation_name,
    medium_parameters,
    medium_statistics,
    my_grid,
    supplied_indices=supplied_indices,
    use_dirac_density=False,
    integration_method="cubature",
    covariance_cubature_scheme=None,
    integration_task_config=use_np_config,
)
integration_result_list, duration = input_statistics_manager.get_statistics()

/tmp/ipykernel_2031537/3981359305.py:5: UserWarning: No parent_data_dir provided. Defaulting to current working directory: /home/nbyrnes/code/random-matrix/data.
  input_statistics_manager = InputStatisticsManager(


[08/18 16:59:28] Classify singles


100%|██████████| 101/101 [00:00<00:00, 3880.70it/s]

[08/18 16:59:28] Done
[08/18 16:59:28] Classify quadruples



100%|██████████| 4/4 [00:00<00:00, 7703.04it/s]

[08/18 16:59:28] Done
[08/18 16:59:28] Prepare covariance tasks



100%|██████████| 4/4 [00:02<00:00,  1.92it/s]


[08/18 16:59:30] Done


In [ ]:
print(integration_result_list.results[0].sub_block_locations)
print(integration_result_list.results[1].sub_block_locations)
print(integration_result_list.results[2].sub_block_locations)
print(integration_result_list.results[3].sub_block_locations)

H1 = integration_result_list.results[0].integral.reshape(4, 4)
H2 = integration_result_list.results[1].integral.reshape(4, 4)
H3 = integration_result_list.results[2].integral.reshape(4, 4)
H4 = integration_result_list.results[3].integral.reshape(4, 4)


[(0, 0, 0, 0)]
[(0, 5, 0, 5)]
[(1, 2, 3, 4)]
[(3, 4, 1, 2)]


In [ ]:
print(np.conj(H3.T))

[[-1.45083263e-09+3.47016107e-12j  1.68586777e-10+1.74873277e-12j
  -9.77721519e-11+2.02706177e-12j -1.44151772e-09+4.05555039e-12j]
 [ 1.42486299e-10+7.65592705e-13j -8.80311154e-12-1.19240982e-13j
   5.47985829e-12-5.63603225e-14j  1.42238659e-10+7.33074165e-13j]
 [-1.75777197e-10+9.78166075e-13j  1.01328574e-11+4.73636958e-14j
  -6.88744455e-12+1.28804484e-13j -1.74951040e-10+1.01091078e-12j]
 [-1.41871588e-09+1.65160696e-12j  1.67027819e-10+1.82045576e-12j
  -9.68038544e-11+1.95586767e-12j -1.40926977e-09+2.23689963e-12j]]


In [ ]:
print(H4)

[[-1.45522474e-09+3.47389576e-12j  1.68806799e-10+1.75263304e-12j
  -9.78528543e-11+2.03113851e-12j -1.44590100e-09+4.05999986e-12j]
 [ 1.42588214e-10+7.67639628e-13j -8.81252417e-12-1.19643051e-13j
   5.48219298e-12-5.64798858e-14j  1.42340730e-10+7.35021310e-13j]
 [-1.75955023e-10+9.80050335e-13j  1.01496727e-11+4.74941381e-14j
  -6.89273985e-12+1.29214204e-13j -1.75125665e-10+1.01289387e-12j]
 [-1.42307147e-09+1.65344573e-12j  1.67246025e-10+1.82446151e-12j
  -9.68824986e-11+1.95983878e-12j -1.41361684e-09+2.23945268e-12j]]


In [ ]:
print( H3 - np.conj(H4.T))

[[ 4.39210930e-12+3.73468752e-15j -1.01915054e-13+2.04692280e-15j
   1.77826652e-13+1.88426027e-15j  4.35559208e-12+1.83877203e-15j]
 [-2.20021843e-13+3.90026195e-15j  9.41263524e-15-4.02069330e-16j
  -1.68152543e-14+1.30442320e-16j -2.18205811e-13+4.00574180e-15j]
 [ 8.07024416e-14+4.07673852e-15j -2.33468619e-15-1.19563249e-16j
   5.29529909e-15+4.09720323e-16j  7.86441283e-14+3.97111354e-15j]
 [ 4.38327830e-12+4.44947306e-15j -1.02070566e-13+1.94714530e-15j
   1.74624695e-13+1.98309181e-15j  4.34707162e-12+2.55304952e-15j]]


In [ ]:
print(H1)

[[ 1.34226811e-07-2.62130305e-13j -2.00969823e-09+5.11051328e-14j
   2.01319561e-09+1.44029043e-13j  1.34536509e-07+1.68450910e-11j]
 [-5.86794206e-10-5.56771388e-13j  1.36800251e-08-6.76486167e-13j
  -1.36882785e-08+2.18076803e-13j -5.52977431e-10+1.28494365e-12j]
 [ 5.73279654e-10+1.29091053e-12j -1.36633525e-08-2.20791305e-13j
   1.36719360e-08+6.79200602e-13j  5.39569268e-10-5.50804488e-13j]
 [ 1.34527055e-07-1.68538719e-11j -2.01121061e-09+1.38824125e-13j
   2.01487260e-09+5.63099465e-14j  1.34842392e-07+2.53349481e-13j]]


In [ ]:
print((np.conj(H1.T)-H1 )/2)
print(np.linalg.norm((np.conj(H1.T)-H1 )/2))

print((np.conj(H2.T)-H2 )/2)
print(np.linalg.norm((np.conj(H2.T)-H2 )/2))

[[ 0.00000000e+00+2.62130305e-13j  7.11452010e-10+2.52833128e-13j
  -7.19957976e-10-7.17469789e-13j -4.72717311e-12+4.39046535e-15j]
 [-7.11452010e-10+2.52833128e-13j  0.00000000e+00+6.76486167e-13j
   1.24630177e-11+1.35725124e-15j -7.29116589e-10-7.11883887e-13j]
 [ 7.19957976e-10-7.17469789e-13j -1.24630177e-11+1.35725124e-15j
   0.00000000e+00-6.79200602e-13j  7.37651666e-10+2.47247271e-13j]
 [ 4.72717311e-12+4.39046535e-15j  7.29116589e-10-7.11883887e-13j
  -7.37651666e-10+2.47247271e-13j  0.00000000e+00-2.53349481e-13j]]
2.049596924701975e-09
[[ 0.00000000e+00-2.32172248e-14j -4.25174291e-12+3.73956579e-14j
   1.14677971e-11+1.06344470e-12j  5.17996776e-13+4.17692290e-14j]
 [ 4.25174291e-12+3.73956579e-14j  0.00000000e+00+4.31139761e-14j
   3.02705202e-13-5.58253265e-14j  1.05689014e-11+1.02298195e-12j]
 [-1.14677971e-11+1.06344470e-12j -3.02705202e-13-5.58253265e-14j
   0.00000000e+00+1.66029523e-13j -2.84088889e-11+1.38000242e-13j]
 [-5.17996776e-13+4.17692290e-14j -1.05689014e

In [ ]:
print(H1)
print(H2)

[[ 1.34226811e-07-2.62130305e-13j -2.00969823e-09+5.11051328e-14j
   2.01319561e-09+1.44029043e-13j  1.34536509e-07+1.68450910e-11j]
 [-5.86794206e-10-5.56771388e-13j  1.36800251e-08-6.76486167e-13j
  -1.36882785e-08+2.18076803e-13j -5.52977431e-10+1.28494365e-12j]
 [ 5.73279654e-10+1.29091053e-12j -1.36633525e-08-2.20791305e-13j
   1.36719360e-08+6.79200602e-13j  5.39569268e-10-5.50804488e-13j]
 [ 1.34527055e-07-1.68538719e-11j -2.01121061e-09+1.38824125e-13j
   2.01487260e-09+5.63099465e-14j  1.34842392e-07+2.53349481e-13j]]
[[ 2.03956814e-10+2.32172248e-14j -2.69863075e-12+3.95186201e-14j
   1.02127611e-11+1.42431610e-12j  5.32492543e-10+5.71867394e-11j]
 [-1.12021166e-11-1.14309936e-13j  1.19348828e-10-4.31139761e-14j
  -3.41593955e-10-3.85518473e-11j -3.09254216e-11-3.65301394e-12j]
 [ 3.31483552e-11-3.55120550e-12j -3.40988544e-10+3.86634980e-11j
   9.97898723e-10-1.66029523e-13j  9.18065667e-11+3.58449642e-14j]
 [ 5.33528537e-10-5.72702778e-11j -9.78761892e-12+1.60705005e-12j
  